# Importamos las librerias a utilizar

In [1]:
!pip install -U langchain-community

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 1.9 MB/s eta 0:00:00


In [2]:
!pip install pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 302.3/302.3 kB 4.8 MB/s eta 0:00:00


# Cargamos el PDF, leemos cada página y almacenamos el texto en una variable

In [4]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Cargar el PDF desde ,las carpetas de Drive
pdf_path = "/content/el principito.pdf"
loader = PyPDFLoader(pdf_path)
documents = loader.load()

# Extraer el texto de cada página
text = "\n".join([doc.page_content for doc in documents])

# Segmentamos el texto en chunks, dando la opción de que el texto en cada chunk se pueda solapar

In [5]:
# Dividir en fragmentos (chunks)
text_splitter = RecursiveCharacterTextSplitter(chunk_size= 500, chunk_overlap = 200)

chunks = text_splitter.split_text(text)  # revisar como usar .split_docuement para que sea compatible con chromaDB

# Creamos una función la cual, usando un modelo de embedding, me convierta texto a vector de números

In [6]:
from sentence_transformers import SentenceTransformer

def text_to_vector(text):

    # Cargar el modelo de embeddings
    model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

    # Convertir el texto en un vector numérico
    vector = model.encode(text)

    return vector

# Haciendo uso de text_to_vector, convertimos cada chunk del pdf en vector y lo almacenamos en una lista

In [13]:
chunk_vectors = [text_to_vector(chunk) for chunk in chunks]
print(len(chunk_vectors))

264


# Construimos un árbol de tipo Ball Tree, que son más eficientes para vectores de grandes dimensiones

In [14]:
from sklearn.neighbors import BallTree
import numpy as np

def build_ball_tree(vectors):
    tree = BallTree(vectors, leaf_size=30)
    return tree

def search_ball_tree(tree, query_vector, k=3):
    distances, indices = tree.query([query_vector], k=k)
    return distances, indices

ballTree = build_ball_tree(chunk_vectors)

# Construimos un árbol de tipo KD Tree para comparar su eficiencia contra el Ball tree

In [15]:
from sklearn.neighbors import KDTree
import numpy as np

def build_kd_tree(vectors):
    tree = KDTree(vectors, leaf_size=30)
    return tree

def search_kd_tree(tree, query_vector, k=3):
    distances, indices = tree.query([query_vector], k=k)
    return distances, indices

kdTree = build_kd_tree(chunk_vectors)

# Prueba; Ingresamos una frase u oración, la convertimos a su versión de vector por medio del modelo de embedding y realizamos la busqueda en el árbol, al final imprimimos el texto ingresado y el texto resultante que tiene mayor similitud

In [16]:
import timeit

text = input("Ingrese una frase: ")
query_vector = text_to_vector(text)

# Medir tiempo para BallTree
start_time = timeit.default_timer()
distancesBall, indicesBall = search_ball_tree(ballTree, query_vector, k=1)
ball_time = timeit.default_timer() - start_time

# Medir tiempo para KDTree
start_time = timeit.default_timer()
distancesKd, indicesKd = search_kd_tree(kdTree, query_vector, k=1)
kd_time = timeit.default_timer() - start_time

# Imprimir resultados
print("\nFrase ingresada: " + text + "\n")
print(f"Frase encontrada por el BallTree: {chunks[indicesBall[0][0]]}")
print(f"Tiempo de búsqueda con BallTree: {ball_time:.6f} segundos\n")

print(f"Frase encontrada por el KDTree: {chunks[indicesKd[0][0]]}")
print(f"Tiempo de búsqueda con KDTree: {kd_time:.6f} segundos\n")

# Comparación
if ball_time < kd_time:
    print("BallTree fue más rápido.")
else:
    print("KDTree fue más rápido.")


Ingrese una frase: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta  Frase ingresada: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta

Frase ingresada: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta  Frase ingresada: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta

Frase encontrada por el BallTree: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta	persona	mayor	vive
en	Francia,	donde	pasa	hambre	y	frío.	Verdaderamente	necesita	consuelo.	Si
todas	esas	excusas	no	bastasen,	bien	puedo	dedicar	este	libro	al	niño	que	una
vez	fue	esta	persona	mayor.	Todos	los	mayores	han	sido	primero	niños.	(Pero
pocos	lo	recuerdan).	Corrijo,	pues,	mi	dedicatoria:
A	LEON	WERTH	CUANDO	ERA	NIÑO
	
	
I
	
Cuando	yo	tenía	seis	años	vi	en	un	libro	sobre	la	selva	virgen	que	se
Tiempo de búsqueda con BallTree: 0.001040 segundos

Frase encontrada por el KDTree: hasta	los	libros	para	niños.	Tengo	una	tercera	excusa:	esta	persona	mayor	vive
en	Francia,	donde	pasa